# project_23_protein_na_binder — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [1]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

Python : 3.11.15
Platform: Linux-6.18.5-x86_64-with-glibc2.39
GPU    : NONE FOUND
Structure prediction on CPU is impractically slow.


## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [2]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

Note: you may need to restart the kernel to use updated packages.
Core install done.


In [3]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

# Environment stamp 2026-06-24T03:17:44 UTC
Bio            1.84


py3Dmol        2.4.0


numpy          2.4.6


pandas         3.0.3
matplotlib     3.11.0


seaborn        0.13.2
tqdm           4.68.3
requests       2.33.1


## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [4]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

Helpers ready: install_colabfold(), install_esmfold().


## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [5]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

seeds set to 0
logged: Ran 00_setup; environment stamped.


## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [6]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

Uncomment to mount Drive and set your working directory.


---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — protein–NA recognition, LigandMPNN-NA theory, target motif

**Standard slot:** *define & explore.* **For Project 23 this means:** choose a **DNA/RNA target
motif**, understand how proteins recognize nucleic acids (and why **LigandMPNN** — which conditions
on nucleic-acid atoms — is the right sequence designer), write down the protein–NA metrics + cutoffs,
and run a deterministic **mock** mini-run as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real campaign (RFdiffusion near the NA + protein–NA
complex modeling) wants an **A100** (see `MANUAL.md §2`); everything here runs on a no-GPU **mock**
backend so you can build the plumbing anywhere, then switch to the real backend on Colab Pro / A100.

> **The one hard truth of this project:** a protein can stick to *any* DNA/RNA backbone (the
> phosphates are negative and generic) without reading the intended bases. **Sequence specificity —
> preferring your motif over a scrambled one — is the real challenge**, and it is harder than
> protein–protein binding. Every result here is reported against a **scrambled-motif** control.

## Protein–nucleic-acid recognition in one screen

Proteins read DNA/RNA through a mix of: **base-specific** contacts (H-bonds / van der Waals from side
chains to the edges of bases, mostly in the **major groove** of DNA), **shape readout** (the
sequence-dependent width/curvature of the groove), and **backbone** contacts to the negatively-charged
phosphates (strong, but **generic** — they do *not* encode specificity). Natural motifs that do this:
helix-turn-helix, zinc fingers, leucine zippers (DNA); RRM, dsRBD, PUF repeats (RNA).

**Why LigandMPNN, not ProteinMPNN.** ProteinMPNN designs a sequence from protein backbone context
only — it is **blind to the nucleic acid**, so it cannot place residues to read specific bases.
**LigandMPNN conditions on non-protein atoms, including DNA/RNA**, so at the interface it chooses
residues that contact the bases/backbone you actually want recognized. That difference is the whole
point of this project; notebook 04 benchmarks the two head-to-head at the interface.

## The protein–NA metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the protein | thermostability / ΔG |
| **pae_interaction** | Å | complex-model error across the **protein–NA interface** (key complex metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| **specificity_score (dScore)** | REU-like | interface score on **scrambled** motif − on **intended** motif (higher = prefers your motif) | a measured ΔΔG |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

We reuse the shared `"binder"` cutoffs (scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10) for the
**confidence** layers, and add a **specificity gate** (dScore ≥ margin) on top — because for protein–NA
binders, *confidence is not specificity*. `pae_interaction` low does **not** mean it binds your motif;
a passing design is a **hypothesis** until an EMSA / fluorescence-anisotropy assay with a
**scrambled-NA control** (notebook 05).

## Setup paths

In [7]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_23_protein_na_binder/notebooks


## 1 · Choose the DNA/RNA target motif

The design target is a **nucleic-acid motif** plus (ideally) a structure of a known protein–NA complex
to scaffold near. Fetch a *candidate* protein–NA complex with `data/download_data.py` (**verify the
accession on RCSB** — protein–DNA *and* protein–RNA complexes exist; pick one that matches your goal),
and **you pick the DNA/RNA target motif** (a transcription-factor box, an operator, an RNA hairpin
sequence, ...) from the complex or the literature.

Below we just *declare* an EXAMPLE DNA motif so the notebook runs end-to-end; **replace it with the
motif you derive and verify**. DNA uses A/C/G/T; RNA uses A/C/G/U. `normalize_motif()` validates the
alphabet so typos fail loudly (it rejects IUPAC ambiguity codes — expand them to a concrete sequence).

In [8]:
import na_binder_tools as nbt

NA_TYPE = "DNA"                         # "DNA" or "RNA" — match your target and complex
# EXAMPLE target motif — VERIFY/REPLACE from your chosen complex/literature (data/README.md).
# A CRE-like 8-bp box, strict ACGT, so the plumbing runs; real numbering/sequence depends on your target.
MOTIF = nbt.normalize_motif("TGACGTCA", NA_TYPE)   # EXAMPLE_DATA placeholder motif
SCRAMBLED = nbt.scramble_motif(MOTIF, seed=0)      # specificity control (same bases, shuffled order)

print("na_type :", NA_TYPE)
print("motif   :", MOTIF, " (EXAMPLE — replace with your verified DNA/RNA target motif)")
print("scramble:", SCRAMBLED, " (specificity control — same composition, different order)")

na_type : DNA
motif   : TGACGTCA  (EXAMPLE — replace with your verified DNA/RNA target motif)
scramble: GGTATCAC  (specificity control — same composition, different order)


## 2 · Mock hello-world: scaffold → LigandMPNN → model → specificity

`scripts/na_binder_tools.py` exposes the whole NA-binder pipeline behind one API:
`scaffold_near_na(...)` (RFdiffusion near the NA), `ligandmpnn_na(...)` (NA-aware sequence design — the
central step), `model_complex(...)` (protein–NA complex modeling), and `motif_specificity(...)`
(intended vs scrambled motif). The **mock** backend is deterministic and GPU-free so you can develop
the plumbing. **Never report mock numbers as real** — they are `SYNTHETIC` by construction.

In [9]:
# One backbone -> two NA-aware sequences -> model the complex -> score specificity. All SYNTHETIC.
backbones = nbt.scaffold_near_na(MOTIF, n=2, tool="mock", na_type=NA_TYPE)
designs = nbt.ligandmpnn_na(backbones[0], MOTIF, n=2, tool="mock", na_type=NA_TYPE, seq_tool="ligandmpnn")
nbt.score_designs(designs, na=MOTIF, tool="mock")          # protein-NA complex -> pae_interaction, plddt, scrmsd
nbt.add_specificity(designs, MOTIF, scrambled=SCRAMBLED, tool="mock")   # intended vs scrambled motif

d = designs[0]
print("example LigandMPNN (NA-aware) design:")
print("  id   :", d.design_id)
print("  len  :", d.length, "aa   na_type:", d.na_type, "  motif:", d.target_motif)
print("  seq  :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd, " plddt =", d.plddt, " (SYNTHETIC)")
print("  dG_motif =", d.dG_motif, " dG_scrambled =", d.dG_scrambled,
      " specificity_score =", d.specificity_score, " is_specific =", d.is_specific, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'rfdiffusion'/'ligandmpnn'/'boltz' on Colab (A100). See MANUAL.md §2.")

example LigandMPNN (NA-aware) design:
  id   : EXAMPLE_DATA_scaffold_0000_seq00_ligandmpnn
  len  : 43 aa   na_type: DNA   motif: TGACGTCA
  seq  : KQHSTQMEKVWEPLWEKGDYPQWNPCHIFGMNKQWITVMIKVD
  pae_interaction = 19.0  scrmsd = 3.0  plddt = 90.0  (SYNTHETIC)
  dG_motif = -36.0  dG_scrambled = -28.0  specificity_score = 8  is_specific = True  (SYNTHETIC)
  synthetic flag  : True -> SYNTHETIC — mock backend, not a real design/prediction

Reminder: switch tool='mock' -> 'rfdiffusion'/'ligandmpnn'/'boltz' on Colab (A100). See MANUAL.md §2.


## 3 · Why the scrambled-motif control matters (specificity proxy)

A binder only **reads your motif** if it scores better on the intended motif than on a scrambled one
(same base composition, different order). `motif_specificity()` returns `dScore = score(scrambled) −
score(intended)`; **positive and large ⇒ prefers your motif ⇒ specific**. A design with great
`pae_interaction` but `dScore ≈ 0` is a **non-specific backbone-gripper** — common, and the honest
hard truth of protein–NA design. This is a *computational proxy*; the wet-lab scrambled-NA
EMSA/anisotropy control (notebook 05) is what actually tests it.

In [10]:
for b in designs:
    tag = "SPECIFIC" if b.is_specific else "non-specific"
    print(f"{b.design_id}: dG_motif={b.dG_motif}  dG_scram={b.dG_scrambled}  "
          f"dScore={b.specificity_score}  -> {tag} (SYNTHETIC)")
print("\nMany designs will be non-specific — that is expected and must be reported honestly.")

EXAMPLE_DATA_scaffold_0000_seq00_ligandmpnn: dG_motif=-36.0  dG_scram=-28.0  dScore=8  -> SPECIFIC (SYNTHETIC)
EXAMPLE_DATA_scaffold_0000_seq01_ligandmpnn: dG_motif=-27.0  dG_scram=-42.0  dScore=-15  -> non-specific (SYNTHETIC)

Many designs will be non-specific — that is expected and must be reported honestly.


## Visualize a protein–NA complex (py3Dmol)

Use this to eyeball a predicted protein–NA complex once you have a real PDB (from Boltz-2 / AF3-style
modeling): protein cartoon + nucleic-acid sticks, so you can see whether the protein sits in the major
groove / on the bases (specific) or just along the backbone (non-specific).

In [11]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})                 # protein
    view.addStyle({"resn": ["DA","DT","DG","DC","A","U","G","C"]},    # nucleic acid
                  {"stick": {}})
    view.zoomTo()
    return view.show()

# Example (after a real Boltz-2 / AF3-style prediction writes a protein-NA complex PDB):
# show_complex("results/model/top_complex.pdb")
print("show_complex(pdb_path) ready (protein cartoon + nucleic-acid sticks).")

show_complex(pdb_path) ready (protein cartoon + nucleic-acid sticks).


## D0 checklist
- [ ] Protein–NA complex accession verified on RCSB (the `data/` candidate is a *candidate* — confirm it is the right protein–DNA/RNA complex, chains, resolution).
- [ ] **DNA/RNA target motif chosen** (from the complex/literature, not invented) and its scrambled control written down.
- [ ] One-paragraph definition of each protein–NA metric **with** its "does not mean" note (esp. confidence ≠ specificity).
- [ ] Reproduced mock mini-run (scaffold → LigandMPNN → model → specificity) with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls (incl. the **scrambled-NA** control); `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — scaffold near the NA + LigandMPNN design around it.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — scaffold near the NA + LigandMPNN design around it

**Standard slot:** *design campaign.* **For Project 23 this is the core:** build the NA-binder pool (D2):
- **RFdiffusion** — scaffold protein backbones **docked against the nucleic-acid target** (hold the NA
  as context so the backbone forms a complementary recognition surface).
- **LigandMPNN (NA-aware)** — design sequences for each backbone **with the NA in context** (the central
  step). We also design a **ProteinMPNN (NA-blind)** set on the *same* backbones for the nb04 benchmark.

Then model every design as a **protein–NA complex** (`pae_interaction` is the key complex metric).

> **Compute honesty:** a real campaign wants an **A100** (Colab Pro+ or a cluster) for RFdiffusion near
> the NA and for protein–NA complex modeling. **LigandMPNN/ProteinMPNN are CPU-cheap** — the modeling
> is the bottleneck. Free **T4** = a *small fallback* (few backbones, small modeling batch). The cells
> below run on the deterministic **mock** backend so the plumbing executes anywhere; the real calls +
> A100 notes are shown alongside. Run `00_setup.ipynb` first.
>
> **Protein–NA design is newer and harder than protein–protein.** RFdiffusion's nucleic-acid support is
> evolving — **verify the current protocol/commit** (version-verify cell below).

## Setup paths

In [12]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_23_protein_na_binder/notebooks


## Version-verify the pinned upstreams (tools change!)

The NA-binder tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log it).
LigandMPNN's nucleic-acid support and RFdiffusion's NA protocol both evolve, so this check matters here.

In [13]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   LigandMPNN   https://github.com/dauparas/LigandMPNN        # NA-aware sequence design (CENTRAL); pin <commit>
#   RFdiffusion  https://github.com/RosettaCommons/RFdiffusion # scaffold near the NA; pin <commit>
#   Boltz        https://github.com/jwohlwend/boltz            # protein-NA complex modeling (Boltz-2); pin <commit>
#   ColabFold    https://github.com/sokrypton/ColabFold        # AF2 complex fallback; pin <commit>
PINNED = {
    "LigandMPNN":  "https://github.com/dauparas/LigandMPNN",
    "RFdiffusion": "https://github.com/RosettaCommons/RFdiffusion",
    "Boltz":       "https://github.com/jwohlwend/boltz",
    "ColabFold":   "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:12s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:12s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("Verify LigandMPNN's NA (ligand_mpnn) model + RFdiffusion's nucleic-acid protocol specifically — they evolve.")

  [200] LigandMPNN   https://github.com/dauparas/LigandMPNN


  [200] RFdiffusion  https://github.com/RosettaCommons/RFdiffusion


  [200] Boltz        https://github.com/jwohlwend/boltz


  [200] ColabFold    https://github.com/sokrypton/ColabFold

Non-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.
Verify LigandMPNN's NA (ligand_mpnn) model + RFdiffusion's nucleic-acid protocol specifically — they evolve.


## 1 · Define the campaign

Same target motif as notebook 01. Set honest campaign sizes; the cells run on `mock` so they execute
anywhere. On Colab (A100) switch the `TOOL_*` to the real backends — and **shrink the numbers on a
T4** (few backbones, a small modeling batch). We generate **both** sequence designers (LigandMPNN
NA-aware + ProteinMPNN NA-blind) on the **same backbones** so the nb04 head-to-head isolates the value
of nucleic-acid conditioning.

In [14]:
import na_binder_tools as nbt
import pandas as pd

NA_TYPE = "DNA"
MOTIF = nbt.normalize_motif("TGACGTCA", NA_TYPE)   # EXAMPLE — replace with your verified target motif
SCRAMBLED = nbt.scramble_motif(MOTIF, seed=0)

# Honest campaign sizes: scaffold many backbones near the NA, design several sequences each.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BACKBONES   = 40      # -> hundreds of RFdiffusion backbones near the NA on A100; few on T4
N_SEQ_PER_BB  = 2       # -> 4-8 LigandMPNN sequences per backbone on Colab

TOOL_SCAFFOLD = "mock"  # -> "rfdiffusion" on Colab (A100)
TOOL_SEQ      = "mock"  # -> "ligandmpnn" / "proteinmpnn" on Colab
TOOL_MODEL    = "mock"  # -> "boltz" / "af3" on Colab (A100)

print(f"backbones (RFdiffusion): n={N_BACKBONES} tool={TOOL_SCAFFOLD}")
print(f"sequences/backbone     : n={N_SEQ_PER_BB} tool={TOOL_SEQ} (LigandMPNN NA-aware + ProteinMPNN NA-blind)")
print(f"complex model          : tool={TOOL_MODEL}")
print("motif :", MOTIF, " scrambled:", SCRAMBLED)

backbones (RFdiffusion): n=40 tool=mock
sequences/backbone     : n=2 tool=mock (LigandMPNN NA-aware + ProteinMPNN NA-blind)
complex model          : tool=mock
motif : TGACGTCA  scrambled: GGTATCAC


## 2 · Scaffold backbones near the nucleic acid (RFdiffusion)

Diffuse protein backbones docked against the NA target, holding the nucleic acid as fixed context so
the backbone forms a complementary recognition surface (a helix into the major groove, a sheet, ...).
Sequence is **not** designed yet. On A100 this is hundreds of backbones; the `mock` backend returns
deterministic `SYNTHETIC` backbones with placeholder sequences.

In [15]:
# Real call (Colab, A100): nbt.scaffold_near_na(MOTIF, n=N_BACKBONES, tool="rfdiffusion", na_type=NA_TYPE)
#   keep the NA chain fixed as context; verify RFdiffusion's current nucleic-acid protocol/commit.
backbones = nbt.scaffold_near_na(MOTIF, n=N_BACKBONES, tool=TOOL_SCAFFOLD, na_type=NA_TYPE)
print(f"scaffolded {len(backbones)} backbones near the {NA_TYPE} target (tool={TOOL_SCAFFOLD}; SYNTHETIC if mock)")
print("example:", backbones[0].design_id, " length=", backbones[0].length,
      " (sequence undesigned ->", backbones[0].seq_tool + ")")

scaffolded 40 backbones near the DNA target (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_scaffold_0000  length= 43  (sequence undesigned -> (undesigned))


## 3 · Design sequences around the NA — LigandMPNN (NA-aware) **and** ProteinMPNN (NA-blind)

The central step. **LigandMPNN** conditions on the nucleic-acid atoms, so its interface residues are
chosen to read the bases/backbone. **ProteinMPNN** sees only the protein backbone (NA-blind) — we design
it on the *same* backbones purely as the benchmark baseline. (Ligand)MPNN is CPU-cheap; this is not the
bottleneck. Then model each design as a protein–NA complex (the slow step on Colab).

In [16]:
# Real call (Colab): nbt.ligandmpnn_na(bb, MOTIF, n=N_SEQ_PER_BB, tool="ligandmpnn", na_type=NA_TYPE,
#   seq_tool="ligandmpnn", temperature=0.1)  with the NA in context (model_type=ligand_mpnn).
#   For the NA-BLIND baseline: tool="proteinmpnn", seq_tool="proteinmpnn".
ligand_designs, protein_designs = [], []
for bb in backbones:
    ligand_designs  += nbt.ligandmpnn_na(bb, MOTIF, n=N_SEQ_PER_BB, tool=TOOL_SEQ,
                                         na_type=NA_TYPE, seq_tool="ligandmpnn")
    protein_designs += nbt.ligandmpnn_na(bb, MOTIF, n=N_SEQ_PER_BB, tool=TOOL_SEQ,
                                         na_type=NA_TYPE, seq_tool="proteinmpnn")

# Model each (protein, NA) complex -> pae_interaction, plddt, scrmsd.
nbt.score_designs(ligand_designs,  na=MOTIF, tool=TOOL_MODEL)
nbt.score_designs(protein_designs, na=MOTIF, tool=TOOL_MODEL)

# Specificity: intended motif vs scrambled motif (the protein-NA-specific layer).
nbt.add_specificity(ligand_designs,  MOTIF, scrambled=SCRAMBLED, tool=TOOL_MODEL)
nbt.add_specificity(protein_designs, MOTIF, scrambled=SCRAMBLED, tool=TOOL_MODEL)

print(f"LigandMPNN (NA-aware) designs: {len(ligand_designs)}")
print(f"ProteinMPNN (NA-blind) designs: {len(protein_designs)}")
print("example LigandMPNN:", ligand_designs[0].design_id,
      "pae_interaction=", ligand_designs[0].pae_interaction,
      "specificity_score=", ligand_designs[0].specificity_score)

LigandMPNN (NA-aware) designs: 80
ProteinMPNN (NA-blind) designs: 80
example LigandMPNN: EXAMPLE_DATA_scaffold_0000_seq00_ligandmpnn pae_interaction= 19.0 specificity_score= 8


## 4 · Assemble + persist both pools

Write one tidy CSV per sequence designer (plus a combined one). These feed notebook 03 (the shared
filter) and notebook 04 (the LigandMPNN-vs-ProteinMPNN benchmark + specificity analysis). We add an
EXAMPLE `solubility` column so the physics layer has something to act on in the dry run — on Colab
these come from the real modeling/energetics; for `mock` they are SYNTHETIC.

In [17]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, seq_tool=d.seq_tool,
            na_type=d.na_type, target_motif=d.target_motif, length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            dG_motif=d.dG_motif, dG_scrambled=d.dG_scrambled,
            specificity_score=d.specificity_score, is_specific=d.is_specific,
            solubility=0.3,        # EXAMPLE_DATA placeholder so Layer 3 (physics) runs in the dry run
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_lig = pool_to_df(ligand_designs);  df_lig.to_csv("results/ligandmpnn_designs.csv", index=False)
df_pro = pool_to_df(protein_designs); df_pro.to_csv("results/proteinmpnn_designs.csv", index=False)
combined = pd.concat([df_lig, df_pro], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/ligandmpnn_designs.csv  ", df_lig.shape, " (NA-aware — your real candidates)")
print("wrote results/proteinmpnn_designs.csv ", df_pro.shape, " (NA-blind baseline — for the benchmark)")
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined.head(4)

wrote results/ligandmpnn_designs.csv   (80, 16)  (NA-aware — your real candidates)
wrote results/proteinmpnn_designs.csv  (80, 16)  (NA-blind baseline — for the benchmark)
wrote results/all_designs.csv          (160, 16)

ALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.


,design_id,paradigm,seq_tool,na_type,target_motif,length,sequence,plddt,pae_interaction,scrmsd,dG_motif,dG_scrambled,specificity_score,is_specific,solubility,synthetic
0,EXAMPLE_DATA_scaffold_0000_seq00_ligandmpnn,ligandmpnn,ligandmpnn,DNA,TGACGTCA,43,KQHSTQMEKVWEPLWEKGDYPQWNPCHIFGMNKQWITVMIKVD,90.0,19.0,3.00,-36.0,-28.0,8,True,0.3,True
1,EXAMPLE_DATA_scaffold_0000_seq01_ligandmpnn,ligandmpnn,ligandmpnn,DNA,TGACGTCA,43,MNALWYALRIPLWYTVMNKCHSAVDSPCRNFLHSFCMEKVWEP,76.0,19.0,1.96,-27.0,-42.0,-15,False,0.3,True
2,EXAMPLE_DATA_scaffold_0001_seq00_ligandmpnn,ligandmpnn,ligandmpnn,DNA,TGACGTCA,80,SAGMSTLMEPVRYFQRSTLMIAVRSPQMSPVRSPGWITQRIKQDIF...,91.0,20.0,3.51,-22.0,-32.0,-10,False,0.3,True
3,EXAMPLE_DATA_scaffold_0001_seq01_ligandmpnn,ligandmpnn,ligandmpnn,DNA,TGACGTCA,80,YTGMEAGWNPGWNTLMSPQWSACMETCHYKLWYTQMSKQMIPVRIF...,80.0,17.0,2.50,-26.0,-35.0,-9,False,0.3,True


## D2 checklist
- [ ] Backbones scaffolded **near the nucleic acid** (hundreds on A100; few on T4), NA held as context.
- [ ] **LigandMPNN (NA-aware)** sequences designed around the NA (the central step) — your real candidate pool.
- [ ] **ProteinMPNN (NA-blind)** designed on the *same* backbones as the benchmark baseline.
- [ ] Every design modeled as a protein–NA complex (`pae_interaction` parsed) **and** scored for motif-vs-scrambled specificity; both pools written to `results/`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured (esp. LigandMPNN NA model + RFdiffusion NA protocol); 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on the pools.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs) + specificity gate

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 23** you build `fp.Design` **binder** objects from the pools, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per sequence designer** (LigandMPNN vs ProteinMPNN) so the head-to-head is fair (D3 part 1). On top of
the shared confidence layers we add a **specificity gate** (prefer the intended motif over a scrambled
one) — because for protein–NA binders, *confidence is not specificity*.

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/ligandmpnn_designs.csv` + `results/proteinmpnn_designs.csv` exist.

## Setup paths

In [18]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_23_protein_na_binder/notebooks


## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. We reuse
the `"binder"` confidence cutoffs (scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10) and add a **specificity gate**
on top (a protein–NA-specific check the shared module does not encode).

In [19]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])
SPEC_MARGIN = 1.5   # min specificity_score (dScore) to call a design motif-specific (tune per project)
print("specificity gate: specificity_score >=", SPEC_MARGIN, "(applied AFTER the shared layers)")

Loaded shared filtering_pipeline from: /home/user/biofx_python/denovo_protein_design_course/shared/filtering_pipeline.py
binder cutoffs: {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}
specificity gate: specificity_score >= 1.5 (applied AFTER the shared layers)


## Build `Design` (binder) objects from the pools

Map each pool row onto `fp.Design` with `design_type="binder"`. The confidence metrics drive the shared
layers: `scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency) and `solubility` (Layer 3
physics). We carry `seq_tool`, `specificity_score`, and `is_specific` in `extra` for the specificity
gate + the nb04 head-to-head. (Mock has no independent orthogonal predictor, so we run Layers 1+3 here;
on Colab add a second protein–NA modeler for Layer 2.)

In [20]:
import os
import pandas as pd

# Regenerate the pools if a fresh session lost them (deterministic mock).
if not (os.path.exists("results/ligandmpnn_designs.csv") and os.path.exists("results/proteinmpnn_designs.csv")):
    import na_binder_tools as nbt
    NA_TYPE = "DNA"; MOTIF = nbt.normalize_motif("TGACGTCA", NA_TYPE); SCRAMBLED = nbt.scramble_motif(MOTIF, seed=0)
    bbs = nbt.scaffold_near_na(MOTIF, n=40, tool="mock", na_type=NA_TYPE)
    lig, pro = [], []
    for bb in bbs:
        lig += nbt.ligandmpnn_na(bb, MOTIF, n=2, tool="mock", na_type=NA_TYPE, seq_tool="ligandmpnn")
        pro += nbt.ligandmpnn_na(bb, MOTIF, n=2, tool="mock", na_type=NA_TYPE, seq_tool="proteinmpnn")
    for grp in (lig, pro):
        nbt.score_designs(grp, na=MOTIF, tool="mock"); nbt.add_specificity(grp, MOTIF, scrambled=SCRAMBLED, tool="mock")
    def _q(designs, p):
        pd.DataFrame([dict(design_id=d.design_id, paradigm=d.paradigm, seq_tool=d.seq_tool,
                           na_type=d.na_type, target_motif=d.target_motif, length=d.length, sequence=d.sequence,
                           plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                           dG_motif=d.dG_motif, dG_scrambled=d.dG_scrambled,
                           specificity_score=d.specificity_score, is_specific=d.is_specific,
                           solubility=0.3, synthetic=d.synthetic) for d in designs]).to_csv(p, index=False)
    _q(lig, "results/ligandmpnn_designs.csv"); _q(pro, "results/proteinmpnn_designs.csv")

df_lig = pd.read_csv("results/ligandmpnn_designs.csv")
df_pro = pd.read_csv("results/proteinmpnn_designs.csv")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd modeler on Colab
        solubility=r.get("solubility", 0.3),
        # NOTE: the shared "binder" cutoffs include rosetta_dG/sc; for protein-NA we leave those None
        # (they apply to protein-protein interfaces) and gate on SPECIFICITY instead, below.
        extra={"seq_tool": r.get("seq_tool"),
               "specificity_score": r.get("specificity_score"),
               "is_specific": bool(r.get("is_specific"))},
    )

binders_lig = [row_to_binder(r) for _, r in df_lig.iterrows()]
binders_pro = [row_to_binder(r) for _, r in df_pro.iterrows()]
print(f"built {len(binders_lig)} LigandMPNN + {len(binders_pro)} ProteinMPNN binder Designs")

built 80 LigandMPNN + 80 ProteinMPNN binder Designs


## Run the pipeline — per sequence designer (fair head-to-head)

`run_pipeline(design_type="binder")` applies the shared confidence cutoffs in order and returns a
ranked DataFrame with survival counts in `df.attrs`. We run **each designer separately** so the
survival funnels are comparable. We use Layers 1+3 here (mock has no independent orthogonal source; add
Layer 2 on Colab with a second protein–NA modeler). The protein–NA `"binder"` physics cutoffs for
`rosetta_dG`/`sc` are left unset (those are protein–protein metrics); **specificity is gated next**.

In [21]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["seq_tool"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, confidence hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_lig = run_one(binders_lig, "ligandmpnn")
ranked_pro = run_one(binders_pro, "proteinmpnn")

ranked = pd.concat([ranked_lig, ranked_pro], ignore_index=True)
# Pull specificity out of `extra` into columns for the gate + nb04.
ranked["specificity_score"] = ranked["extra"].apply(lambda e: (e or {}).get("specificity_score"))
ranked["is_specific"] = ranked["extra"].apply(lambda e: bool((e or {}).get("is_specific")))
# Carry target_motif / na_type from the pools so they flow into top_candidates.csv (for the nb05 plan).
_meta = pd.concat([df_lig, df_pro], ignore_index=True).set_index("design_id")
for col in ("target_motif", "na_type"):
    ranked[col] = ranked["design_id"].map(_meta[col])
ranked = ranked.sort_values(["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "seq_tool", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "specificity_score", "is_specific"]]

ligandmpnn  : 80 designs, survival {'L1': 11, 'L3': 11}, confidence hit rate = 11/80 (13.8%)
proteinmpnn : 80 designs, survival {'L1': 12, 'L3': 12}, confidence hit rate = 12/80 (15.0%)

wrote results/all_ranked.csv (160, 25)


,design_id,seq_tool,layers_passed,score,scrmsd,plddt,pae_interaction,specificity_score,is_specific
0,EXAMPLE_DATA_scaffold_0024_seq00_proteinmpnn,proteinmpnn,3,3.7500,0.89,89.0,6.0,5,True
1,EXAMPLE_DATA_scaffold_0034_seq00_ligandmpnn,ligandmpnn,3,3.7333,0.92,82.0,5.0,-16,False
2,EXAMPLE_DATA_scaffold_0030_seq01_proteinmpnn,proteinmpnn,3,3.6500,0.97,87.0,6.0,3,True
3,EXAMPLE_DATA_scaffold_0014_seq01_proteinmpnn,proteinmpnn,3,3.5833,0.85,85.0,8.0,3,True
4,EXAMPLE_DATA_scaffold_0013_seq01_proteinmpnn,proteinmpnn,3,3.5500,1.07,87.0,6.0,4,True
5,EXAMPLE_DATA_scaffold_0022_seq01_ligandmpnn,ligandmpnn,3,3.4333,1.26,86.0,5.0,11,True
6,EXAMPLE_DATA_scaffold_0027_seq01_proteinmpnn,proteinmpnn,3,3.4167,0.93,93.0,10.0,-9,False
7,EXAMPLE_DATA_scaffold_0007_seq00_proteinmpnn,proteinmpnn,3,3.2000,1.28,98.0,9.0,6,True
8,EXAMPLE_DATA_scaffold_0039_seq00_ligandmpnn,ligandmpnn,3,3.1000,1.28,88.0,9.0,-12,False
9,EXAMPLE_DATA_scaffold_0009_seq00_proteinmpnn,proteinmpnn,3,2.9167,1.35,85.0,10.0,-4,False


## The specificity gate (protein–NA-specific, on top of the shared layers)

Confidence layers say the model is *sure where* the protein sits; they do **not** say it reads your
motif. The specificity gate keeps only designs that (a) pass the shared confidence layers **and** (b)
prefer the intended motif over the scrambled one (`specificity_score ≥ margin`). This funnel — confident
**and** specific — is the real protein–NA hit set. Expect a steep drop here: most confident designs are
non-specific backbone-grippers.

In [22]:
conf = ranked[ranked["layers_passed"] >= 3]
spec = conf[conf["specificity_score"] >= SPEC_MARGIN]
print(f"confident (passed shared layers): {len(conf)}/{len(ranked)}")
print(f"confident AND motif-specific    : {len(spec)}/{len(ranked)}  "
      f"({100*len(spec)/max(len(ranked),1):.1f}%)   [SYNTHETIC if mock]")
for label, g in spec.groupby("seq_tool"):
    print(f"   {label:12s}: {len(g)} confident+specific designs")
spec.to_csv("results/specific_candidates.csv", index=False)
print("wrote results/specific_candidates.csv (the real protein-NA hit set)")

confident (passed shared layers): 23/160
confident AND motif-specific    : 11/160  (6.9%)   [SYNTHETIC if mock]
   ligandmpnn  : 4 confident+specific designs
   proteinmpnn : 7 confident+specific designs
wrote results/specific_candidates.csv (the real protein-NA hit set)


## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **combined**
pool for one comparable figure; the per-designer runs above are the rigorous version. Read the bars as
a funnel: steep drops show which layer discriminates (and remember the **specificity gate** above is the
protein–NA-specific drop the shared report does not draw).

In [23]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_lig + binders_pro
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p23")
print("\nsaved results/p23_survival.png + results/p23_ranked.csv")
top

Total designs: 160
  L1 survivors: 23  (14.4%)
  L3 survivors: 23  (14.4%)



saved results/p23_survival.png + results/p23_ranked.csv


,design_id,design_type,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG,tm_to_pdb
0,EXAMPLE_DATA_scaffold_0024_seq00_proteinmpnn,binder,3,3.7500,0.89,89.0,6.0,None,None
1,EXAMPLE_DATA_scaffold_0034_seq00_ligandmpnn,binder,3,3.7333,0.92,82.0,5.0,None,None
2,EXAMPLE_DATA_scaffold_0030_seq01_proteinmpnn,binder,3,3.6500,0.97,87.0,6.0,None,None
3,EXAMPLE_DATA_scaffold_0014_seq01_proteinmpnn,binder,3,3.5833,0.85,85.0,8.0,None,None
4,EXAMPLE_DATA_scaffold_0013_seq01_proteinmpnn,binder,3,3.5500,1.07,87.0,6.0,None,None
5,EXAMPLE_DATA_scaffold_0022_seq01_ligandmpnn,binder,3,3.4333,1.26,86.0,5.0,None,None
6,EXAMPLE_DATA_scaffold_0027_seq01_proteinmpnn,binder,3,3.4167,0.93,93.0,10.0,None,None
7,EXAMPLE_DATA_scaffold_0007_seq00_proteinmpnn,binder,3,3.2000,1.28,98.0,9.0,None,None
8,EXAMPLE_DATA_scaffold_0039_seq00_ligandmpnn,binder,3,3.1000,1.28,88.0,9.0,None,None
9,EXAMPLE_DATA_scaffold_0009_seq00_proteinmpnn,binder,3,2.9167,1.35,85.0,10.0,None,None


## Honest hit-rate accounting (per designer, confidence vs specificity)

Report `N passing confidence layers / N generated` **and** `N confident-AND-specific / N generated`,
for **each** designer — these are the numbers the nb04 head-to-head builds on. The gap between them is
the specificity problem, quantified. Remember: survival is *enrichment*, not *correctness*. Mock numbers
are SYNTHETIC.

In [24]:
for label, df in [("ligandmpnn", ranked_lig), ("proteinmpnn", ranked_pro)]:
    n = len(df); conf_n = int((df["layers_passed"] >= 3).sum())
    spec_n = int(spec[spec["seq_tool"] == label].shape[0])
    print(f"{label:12s}: confident = {conf_n}/{n} ({100*conf_n/max(n,1):.1f}%)  |  "
          f"confident+specific = {spec_n}/{n} ({100*spec_n/max(n,1):.1f}%)   [SYNTHETIC if mock]")
print("\nThe confident->specific drop IS the protein-NA challenge. Report both numbers, not just the best design.")

ligandmpnn  : confident = 11/80 (13.8%)  |  confident+specific = 4/80 (5.0%)   [SYNTHETIC if mock]
proteinmpnn : confident = 12/80 (15.0%)  |  confident+specific = 7/80 (8.8%)   [SYNTHETIC if mock]

The confident->specific drop IS the protein-NA challenge. Report both numbers, not just the best design.


## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per sequence designer** (funnel figure `results/p23_survival.png`).
- [ ] **Specificity gate** applied on top (`results/specific_candidates.csv`): confident **AND** prefers the intended motif.
- [ ] Honest hit-rate accounting: confidence hit rate **and** confident-AND-specific rate for LigandMPNN and ProteinMPNN.
- [ ] Mapping assumptions (which fields → which `Design` attributes; why rosetta_dG/sc left unset for protein–NA) written down.

**Next:** `04_validate.ipynb` — LigandMPNN-vs-ProteinMPNN benchmark + specificity figures + complex modeling.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — LigandMPNN vs ProteinMPNN + sequence-specificity analysis

**Standard slot:** *validate (in silico).* **For Project 23 this is the core comparison:** the
head-to-head between **LigandMPNN (NA-aware)** and **ProteinMPNN (NA-blind)** at the interface, the
**sequence-specificity analysis** (intended motif vs scrambled motif), and protein–NA complex-modeling
figures, with publication-style plots (D3 part 2).

The thesis of this notebook: **the value of LigandMPNN is specificity, not just confidence.** ProteinMPNN
can produce confident-looking designs, but because it is blind to the nucleic acid it should be **less
likely to prefer the intended motif** over a scramble. We test exactly that.

Needs `results/ligandmpnn_designs.csv` + `results/proteinmpnn_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

## Setup paths

In [25]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_23_protein_na_binder/notebooks


## 1 · Head-to-head: confidence hit rate **and** specificity rate

Compare the two designers on (a) confidence hit rate (passed the shared layers), (b) **specificity rate**
(confident AND `specificity_score ≥ margin`), and (c) the `pae_interaction` distribution. A fair
comparison filters both identically (notebook 03) and reports the *distribution*, not the single best.
Mock numbers are SYNTHETIC.

In [26]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
SPEC_MARGIN = 1.5
print("sequence designers:", ranked["seq_tool"].value_counts().to_dict())

summary = []
for t, g in ranked.groupby("seq_tool"):
    n = len(g)
    conf = int((g["layers_passed"] >= 3).sum())
    spec = int(((g["layers_passed"] >= 3) & (g["specificity_score"] >= SPEC_MARGIN)).sum())
    summary.append(dict(seq_tool=t, n=n, confident=conf, confident_specific=spec,
                        confidence_rate_pct=round(100*conf/max(n,1), 1),
                        specificity_rate_pct=round(100*spec/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_dScore=round(float(g["specificity_score"].median()), 2)))
summary = pd.DataFrame(summary)
print("\nhead-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))
print("\nKEY READOUT: compare the SPECIFICITY rate (NA-aware LigandMPNN should win there if NA conditioning helps).")

sequence designers: {'proteinmpnn': 80, 'ligandmpnn': 80}

head-to-head summary (SYNTHETIC if mock):
   seq_tool  n  confident  confident_specific  confidence_rate_pct  specificity_rate_pct  median_pae  median_dScore
 ligandmpnn 80         11                   4                 13.8                   5.0        12.0            0.0
proteinmpnn 80         12                   7                 15.0                   8.8        11.0            1.0

KEY READOUT: compare the SPECIFICITY rate (NA-aware LigandMPNN should win there if NA conditioning helps).


In [27]:
# (a) specificity_score distribution per designer; (b) pae_interaction distribution per designer.
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for t, g in ranked.groupby("seq_tool"):
    ax[0].hist(g["specificity_score"].dropna(), bins=15, alpha=0.5, label=t)
    ax[1].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=t)
ax[0].axvline(SPEC_MARGIN, ls="--", color="k", lw=1, label=f"spec margin ({SPEC_MARGIN})")
ax[0].set_xlabel("specificity_score = score(scrambled) - score(motif)\n(higher = prefers intended motif)")
ax[0].set_ylabel("designs"); ax[0].set_title("Sequence specificity"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("pae_interaction (Å, lower better)"); ax[1].set_title("Protein-NA complex confidence"); ax[1].legend(fontsize=8)
fig.suptitle("LigandMPNN (NA-aware) vs ProteinMPNN (NA-blind) (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p23_headtohead.png", dpi=150); plt.show()
print("saved results/p23_headtohead.png")

saved results/p23_headtohead.png


## 2 · The specificity analysis, made explicit

This is the scientific heart of the project. For each design we compare the interface score on the
**intended motif** vs a **scrambled motif** (same base composition, different order). A point on the
diagonal is **non-specific** (reads the backbone, not the bases); a point below the diagonal **prefers
the intended motif** (specific). Color by designer to see whether nucleic-acid conditioning (LigandMPNN)
pushes designs off the diagonal. Mock numbers are SYNTHETIC — but the *shape* of this analysis is exactly
what you produce on Colab.

In [28]:
lig = pd.read_csv("results/ligandmpnn_designs.csv")
pro = pd.read_csv("results/proteinmpnn_designs.csv")
pools = pd.concat([lig, pro], ignore_index=True)

fig, ax = plt.subplots(figsize=(5.2, 5))
for t, g in pools.groupby("seq_tool"):
    ax.scatter(g["dG_motif"], g["dG_scrambled"], s=18, alpha=0.6, label=t)
lo = float(np.nanmin(pools[["dG_motif", "dG_scrambled"]].values))
hi = float(np.nanmax(pools[["dG_motif", "dG_scrambled"]].values))
ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="non-specific (diagonal)")
ax.set_xlabel("interface score vs INTENDED motif (lower = stronger)")
ax.set_ylabel("interface score vs SCRAMBLED motif")
ax.set_title("Specificity: below the diagonal = prefers the intended motif\n(EXAMPLE_DATA if mock)")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("results/p23_specificity.png", dpi=150); plt.show()
print("saved results/p23_specificity.png")
print("Points ON the diagonal are NON-SPECIFIC backbone-grippers — the honest hard truth of protein-NA design.")

saved results/p23_specificity.png
Points ON the diagonal are NON-SPECIFIC backbone-grippers — the honest hard truth of protein-NA design.


## 3 · Novelty `[extension]`

Novelty = TM-score of each protein backbone to its nearest natural fold (Foldseek/TM-align; `< 0.5` ≈
novel). On Colab, compute it per design and compare the two designers' novelty distributions. Here we
scaffold the analysis (mock has no real structures), so we just show where it plugs in.

In [29]:
# Scaffold: on Colab, run Foldseek/TM-align on each predicted protein backbone -> tm_to_pdb,
# then compare distributions across designers (novel == tm_to_pdb < 0.5).
if "tm_to_pdb" in ranked.columns and ranked["tm_to_pdb"].notna().any():
    for t, g in ranked.groupby("seq_tool"):
        novel = (g["tm_to_pdb"] < 0.5).mean()
        print(f"{t:12s}: novel fraction (TM<0.5) = {novel:.2f}")
else:
    print("Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare designers.")

Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare designers.


## 4 · CRISPR-modulator framing `[extension]`

A topical extension: instead of (or in addition to) a free nucleic-acid motif, target a **Cas protein
surface** to build a *CRISPR modulator* — a mini-protein that tunes or blocks Cas activity (an
anti-CRISPR-like modulator for safer gene editing). That is a **protein–protein** binder problem
(reuse the Project 06 workflow against a Cas target), optionally combined with NA context if the Cas–
guide–DNA complex is the target. Keep the framing **therapeutic / basic-science** (control/safety of
editing), per Responsible Research. Here we just flag where it plugs in.

In [30]:
# Scaffold ONLY. CRISPR-modulator extension = a protein-protein binder vs a Cas surface
# (reuse projects/project_06_pdl1_binder/scripts/binder_tools.py against a Cas target PDB),
# optionally with the guide-RNA/target-DNA as NA context (this project's na_binder_tools).
# Responsible Research: frame as editing CONTROL/safety (an anti-CRISPR-like modulator), not harm.
print("CRISPR-modulator framing is an [extension] scaffold: bind a Cas surface to MODULATE editing.")
print("Treat it as a protein-protein binder (Project 06 workflow) +/- NA context; keep the framing therapeutic.")

CRISPR-modulator framing is an [extension] scaffold: bind a Cas surface to MODULATE editing.
Treat it as a protein-protein binder (Project 06 workflow) +/- NA context; keep the framing therapeutic.


## 5 · Select the top specific candidates per designer

The D★ deliverable wants designs that are **confident AND specific**. Rank the confident-and-specific
set by the composite score and, as a tie-breaker, prefer a higher `specificity_score`. Save the
shortlist for the validation plan (notebook 05).

In [31]:
top_per = []
for t, g in ranked.groupby("seq_tool"):
    g2 = g[(g["layers_passed"] >= 3) & (g["specificity_score"] >= SPEC_MARGIN)].sort_values(
        ["score", "specificity_score"], ascending=False).head(15)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True) if top_per else pd.DataFrame()
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top confident+specific per designer)")
if len(top):
    print(top.groupby("seq_tool").size().to_dict())
    print(top.head(8)[["design_id", "seq_tool", "score", "pae_interaction", "specificity_score"]].to_string(index=False))
else:
    print("No confident+specific designs in this mock run — that is a legitimate (and common) outcome to report.")

wrote results/top_candidates.csv: (11, 25) (top confident+specific per designer)
{'ligandmpnn': 4, 'proteinmpnn': 7}
                                   design_id    seq_tool  score  pae_interaction  specificity_score
 EXAMPLE_DATA_scaffold_0022_seq01_ligandmpnn  ligandmpnn 3.4333              5.0                 11
 EXAMPLE_DATA_scaffold_0011_seq00_ligandmpnn  ligandmpnn 2.3333              5.0                 22
 EXAMPLE_DATA_scaffold_0018_seq00_ligandmpnn  ligandmpnn 2.3333              5.0                  6
 EXAMPLE_DATA_scaffold_0028_seq00_ligandmpnn  ligandmpnn 2.1167             10.0                  6
EXAMPLE_DATA_scaffold_0024_seq00_proteinmpnn proteinmpnn 3.7500              6.0                  5
EXAMPLE_DATA_scaffold_0030_seq01_proteinmpnn proteinmpnn 3.6500              6.0                  3
EXAMPLE_DATA_scaffold_0014_seq01_proteinmpnn proteinmpnn 3.5833              8.0                  3
EXAMPLE_DATA_scaffold_0013_seq01_proteinmpnn proteinmpnn 3.5500              6.0   

## D3 (part 2) checklist
- [ ] Head-to-head: confidence rate **and** specificity rate per designer (figure `results/p23_headtohead.png`).
- [ ] **Specificity analysis** (intended vs scrambled motif) plotted (`results/p23_specificity.png`); non-specific designs called out.
- [ ] Novelty compared across designers (TM-score to PDB) — or the scaffold wired up on Colab.
- [ ] (Extension) CRISPR-modulator framing noted; (extension) the LigandMPNN-vs-ProteinMPNN interface comparison discussed.
- [ ] `results/top_candidates.csv`: top **confident + specific** designs, ready for the validation plan.
- [ ] Honest discussion: confidence ≠ specificity; many designs grip the backbone non-specifically.

**Next:** `05_validation_plan.ipynb` — the EMSA / fluorescence-anisotropy plan with scrambled-NA controls.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — EMSA / fluorescence-anisotropy + scrambled-NA controls

**Standard slot:** *validation plan.* **For Project 23 this means:** turn the top specific candidates
into a **costed, controlled wet-lab plan** — an **EMSA (gel-shift)** and/or **fluorescence-anisotropy**
binding assay vs the target nucleic acid, the mandatory controls (**scrambled-NA negative**,
catalytic/interface dead-mutant of your own design, unrelated protein), an expression strategy, and the
**CRISPR-modulator** stretch (D4/D5).

A design that passes every filter is a **hypothesis** — an EMSA / anisotropy titration with a
**scrambled-NA control** is what tests both *binding* and *specificity*. Needs
`results/top_candidates.csv` (notebook 04). **No fabricated K_D anywhere** — report measured numbers
only after you measure them.

## Setup paths

In [32]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_23_protein_na_binder/notebooks


## 1 · Draft the experimental validation plan

Generate a plan card from the top candidates: assays, controls, expression, timeline, costed reagents.
Fill the `<...>` from your own numbers; this is the deliverable other people will actually read. The
non-negotiables for protein–NA: a **scrambled-NA control** and a **dead-mutant control** (your own
design with its interface residues mutated), because specificity is the whole game.

In [33]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_tool = top.groupby("seq_tool").size().to_dict() if n_top else {}
motif = str(top["target_motif"].iloc[0]) if (n_top and "target_motif" in top.columns) else "<your motif>"

plan = f"""# Protein-NA Binder Validation Plan (Project 23 - by <your name>, <date>)

## Candidates
Top {n_top} confident-AND-specific candidates carried forward ({by_tool}); see results/top_candidates.csv.
Target nucleic acid: {motif} (na_type from your design). EVERY in-silico number is a HYPOTHESIS until
measured - pae_interaction is confidence, specificity_score is a computational proxy, NEITHER is a K_D.

## Expression / reagents
- Protein binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (40-90 aa) -> high yield expected.
- Target nucleic acid: synthesize the {motif} oligo (DNA: HPLC-purified duplex; RNA: in-vitro transcribed
  or synthesized, with a 5' fluorophore (FAM/Cy5) for anisotropy). Order a SCRAMBLED-NA oligo of the SAME
  length/base-composition as the specificity control.

## Assays (go/no-go -> binding -> specificity)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse? not aggregated?).
2. Binding: EMSA (gel-shift) titration of protein vs labeled target NA -> apparent affinity from the shift;
   AND/OR fluorescence anisotropy/polarization titration (labeled NA, protein dilution series) -> apparent K_D.
3. SPECIFICITY (the point): repeat the SAME titration against the SCRAMBLED-NA control. A specific binder
   shifts/binds the intended motif at much lower protein concentration than the scramble. Report the RATIO,
   not just the intended-motif number.
4. Stability: DSF (Tm) of the protein. Deep (optional): co-crystal / cryo-EM of the protein-NA complex;
   competition with a known motif-binding protein.

## Controls (MANDATORY)
- Negative (scrambled-NA): the SAME assay vs a scrambled motif (same composition) -> a specific binder
  should bind it MUCH more weakly. This is the cleanest specificity control and is REQUIRED.
- Negative (dead-mutant): YOUR OWN top design with its predicted NA-interface residues mutated (e.g. the
  base-reading residues -> Ala) -> must LOSE binding to the intended motif.
- Negative (unrelated protein): an unrelated protein of similar size/charge -> should not shift the NA.
- Positive: a known binder of {motif} (the natural protein / a published designed binder, if available)
  to confirm the labeled NA reagent and the assay are working.

## Realistic expectations
Protein-NA design is NEWER and HARDER than protein-protein; SEQUENCE SPECIFICITY is the main failure mode
(designs grip the generic phosphate backbone, not the bases). Expect many in-silico hits to bind
non-specifically or not at all. Report the experimental hit rate AND the specificity ratio honestly.
Do NOT imply a working binder or fabricate a K_D.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + dead-mutant negatives): $<...>, <...> weeks (IGSC-screened provider).
- Labeled target NA + scrambled-NA control oligos (+ unlabeled for EMSA): $<...>.
- Anisotropy plate reader / EMSA gel time + a positive-control reagent: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Designed nucleic-acid-binding proteins for gene-editing modulation / RNA-targeting therapeutics /
synthetic transcription factors (therapeutic / basic-science; low dual-use). Default neutralizing/
therapeutic framing. Gene synthesis via a biosecurity-screening provider; wet lab under institutional
biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md - fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

wrote results/validation_plan.md - fill the <...> placeholders from your numbers.
# Protein-NA Binder Validation Plan (Project 23 - by <your name>, <date>)

## Candidates
Top 11 confident-AND-specific candidates carried forward ({'ligandmpnn': 4, 'proteinmpnn': 7}); see results/top_candidates.csv.
Target nucleic acid: TGACGTCA (na_type from your design). EVERY in-silico number is a HYPOTHESIS until
measured - pae_interaction is confidence, specificity_score is a computational proxy, NEITHER is a K_D.

## Expression / reagents
- Protein binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (40-90 aa) -> high yield expected.
- Target nucleic acid: synthesize the TGACGTCA oligo (DNA: HPLC-purified duplex; RNA: in-vitro transcribed
  or synthesized, wit ...


## 2 · Build the scrambled-NA + dead-mutant negative controls

Two specificity controls, generated alongside the real designs so the EMSA/anisotropy comparison is
airtight:
1. **Scrambled-NA**: the same motif with its bases shuffled (same composition) — order this oligo and
   run the *same* assay; a specific binder should bind it much more weakly.
2. **Dead-mutant**: your own top design with its predicted NA-interface residues mutated — it should
   lose binding to the intended motif. Here we scaffold a sequence-level mutant deterministically; on
   Colab, mutate the *predicted interface* residues specifically.

In [34]:
import random
import na_binder_tools as nbt   # nbt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

MOTIF = str(top["target_motif"].iloc[0]) if (n_top and "target_motif" in top.columns) else "TGACGTCA"
NA_TYPE = str(top["na_type"].iloc[0]) if (n_top and "na_type" in top.columns) else "DNA"

# (1) Scrambled-NA control oligo (order this; same composition, shuffled order).
scrambled_na = nbt.scramble_motif(nbt.normalize_motif(MOTIF, NA_TYPE), seed=0)
print(f"scrambled-NA control oligo: {MOTIF} -> {scrambled_na}  (order BOTH; run the SAME assay)")

# (2) Dead-mutant protein controls (mutate a fraction of residues as a NEGATIVE-CONTROL stand-in).
def dead_mutant(seq, frac=0.4, seed=0):
    """Deterministically mutate a fraction of residues -> Ala/Gly as a NEGATIVE-CONTROL stand-in.
    On Colab, mutate the PREDICTED NA-interface residues specifically (the base-reading positions)."""
    rng = random.Random(seed)
    seq = list(seq); idx = list(range(len(seq))); rng.shuffle(idx)
    for i in idx[:max(1, int(len(seq) * frac))]:
        seq[i] = "A" if rng.random() < 0.5 else "G"
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s and set(s) <= set("ACDEFGHIKLMNPQRSTVWY"):
            negs.append(dict(design_id=str(r["design_id"]) + "_DEADMUT",
                             parent=r["design_id"], seq_tool=r.get("seq_tool"),
                             sequence=dead_mutant(s, seed=nbt._hashints(r["design_id"]) % 10**6),
                             role="NA-interface dead-mutant negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} dead-mutant negatives (+ the scrambled-NA oligo above)")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

scrambled-NA control oligo: TGACGTCA -> GGTATCAC  (order BOTH; run the SAME assay)
wrote results/negative_controls.csv: 11 dead-mutant negatives (+ the scrambled-NA oligo above)


## 3 · (Stretch) CRISPR-modulator validation `[stretch]`

If you took the CRISPR-modulator extension (bind a Cas surface to tune/block editing), the validation is
a **protein–protein** plan (SPR/BLI vs the Cas target + a *functional editing assay*: does the modulator
reduce/redirect Cas cleavage in vitro or in cells?), with the same control discipline (scrambled-interface
negative, unrelated protein). Frame it as editing **control/safety** (an anti-CRISPR-like modulator),
per Responsible Research — never as a tool to defeat safeguards for harm.

In [35]:
# Scaffold ONLY. CRISPR-modulator validation = SPR/BLI vs the Cas target + an in-vitro/cell EDITING assay
#   (does the modulator reduce/redirect Cas cleavage?), with scrambled-interface + unrelated-protein controls.
#   Reuse the Project 06 SPR/BLI plan structure; keep the framing therapeutic/safety (editing CONTROL).
print("CRISPR-modulator validation is a STRETCH scaffold: SPR/BLI + a functional editing-control assay.")
print("Frame as safer/controllable editing (anti-CRISPR-like modulator) - never to defeat safeguards for harm.")

CRISPR-modulator validation is a STRETCH scaffold: SPR/BLI + a functional editing-control assay.
Frame as safer/controllable editing (anti-CRISPR-like modulator) - never to defeat safeguards for harm.


## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **EMSA / fluorescence-anisotropy** titration + **scrambled-NA** repeat, expression, timeline, costed reagents.
- [ ] Controls specified: **scrambled-NA** negative (required), **dead-mutant** negative (`results/negative_controls.csv`), unrelated-protein negative, positive (known motif binder).
- [ ] Specificity reported as a **ratio** (intended vs scrambled), not just the intended-motif number; **no fabricated K_D**.
- [ ] (Stretch) CRISPR-modulator validation scaffolded as a protein–protein + functional-editing plan, framed as editing control/safety.
- [ ] Honest framing: every design is a hypothesis until EMSA/anisotropy; protein–NA specificity is the main failure mode; report the experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a complete protein–NA-binder DBTL turn, honestly reported, with specificity at its center.